<a href="https://colab.research.google.com/github/GaganJangid63/RAG-PROJECT/blob/main/RAG_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
#step 1 Install the tools
!pip install -q transformers sentence-transformers faiss-cpu pypdf python-docx nltk gradio pymupdf

In [44]:
#step 2 set up all required libraries

import os,io,math,gc,json,textwrap,re,random,collections   # Basic Python libraries for file handling, memory, text processing, math, JSON, and random operations

from dataclasses import dataclass   # Helps create simple data classes
from typing import List, Dict, Tuple   # Used for type hints to make code easier to understand

import nltk   # Library for Natural Language Processing (NLP)
nltk.download('punkt')   # Download tokenizer for splitting text into sentences
nltk.download('punkt_tab')   # Download additional tokenizer data
from nltk.tokenize import sent_tokenize   # Function to split paragraphs into sentences

from google.colab import files   # Upload files in Google Colab
from datetime import datetime   # Get current date and time

from pypdf import PdfReader   # Read and extract text from PDF files
from docx import Document as DocxDocuments   # Read Microsoft Word (.docx) files

from sentence_transformers import SentenceTransformer   # Convert text into embeddings (vectors)
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,pipeline   # Load tokenizer, AI model, and pipeline for NLP tasks

import fitz #PymuPDF   # Extract text, images, and layout information from PDF files
import faiss   # Perform fast similarity search on embeddings
import numpy as np   # Numerical operations and array handling
import gradio as gr   # Create a simple web interface for the project
import torch   # Deep learning library used to run AI models

DEVICE= "cuda" if torch.cuda.is_available() else "cpu"   # Use GPU if available, otherwise use CPU
DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU"   # Display which device is being used






[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


('cuda', 'Tesla T4')

In [ ]:
from google.colab import files #uploaded selected files in google colab

uploaded = files.upload()

In [ ]:
uploaded

In [ ]:
#step 3  Uploading your documents with limits
MAX_FILE = 5 # maximub file upload limit
MAX_MB_PER_FILE = 15 #Max size 15mb per file
ALLOWED_EXTS = {'.pdf','.txt','.docx'}  #Which files are allow to upload in project


In [ ]:
#step 4 Validate and accept your uploaded files

def ext_of(name):
  return os.path.splitext(name)[1].lower()   # Find the file type (extension) like .pdf or .docx

assert len(uploaded) > 0, "No file uploaded"   # Stop the program if no file is uploaded
assert len(uploaded) <= MAX_FILE, f"plese uploaded at most {MAX_FILE} files."   # Allow only the maximum number of files

docs_raw = []   # Create an empty list to keep all valid files

for fname, b in uploaded.items():

    assert ext_of(fname) in ALLOWED_EXTS, f"Unsupported file type for {fname}. Allowed: {ALLOWED_EXTS}"   # Accept only supported file formats

    size_mb = len(b) / (1024*1024)   # Calculate the file size in MB

    assert size_mb <= MAX_MB_PER_FILE, f"{fname} is {size_mb:.2f} MB, exceeds {MAX_MB_PER_FILE} MB."   # Make sure the file is not too large

    docs_raw.append((fname, b))   # Save the valid file name and its data for later use


In [ ]:
#step 5 Function to convert uploaded text file bytes into readable text
def read_txt(bytes_blob: bytes) -> str:
  return io.BytesIO(bytes_blob).read().decode('utf-8', errors='ignore')

In [ ]:
#Read txt file and Extract the data
read_txt(docs_raw[2][1])

In [ ]:
#step 6 Write a function that reads a PDF (given as bytes) and returns all its text

def read_pdf(bytes_blob: bytes) -> str:   # Create a function to read a PDF file and return its text

  reader = PdfReader(io.BytesIO(bytes_blob))   # Open the PDF from its byte data

  texts = []   # Create an empty list to store text from each page

  for page in reader.pages:   # Go through every page in the PDF

     try:
         texts.append(page.extract_text() or "")   # Extract text from the page and save it (use empty text if nothing is found)

     except:
         texts.append("")   # If any page cannot be read, save an empty string instead of stopping the program

  return "\n".join(texts)   # Combine all page texts into one single string and return it

In [ ]:
read_pdf(docs_raw[0][1])   # Read the first uploaded PDF and extract all its text

In [ ]:
#step 7 Write a function that reads a Word (.docx) file from bytes and returns its text

def read_docx(bytes_blob:bytes) -> str:   # Create a function to read a Word (.docx) file and return all its text
  fh = io.BytesIO(bytes_blob)   # Convert the file bytes into a file-like object
  doc = DocxDocuments(fh)   # Open the Word document
  return "\n".join([p.text for p in doc.paragraphs])   # Combine the text from all paragraphs into one string and return it

In [ ]:
read_docx(docs_raw[1][1])   # Read the second uploaded docx and extract all its text

In [ ]:
#step 8 Write a function that picks the right reader based on file extension and returns the text

def load_text_by_ext(fname: str, blob: bytes) -> str:   # Create a function to read a file based on its extension
    ext = os.path.splitext(fname)[1].lower()   # Get the file extension (like .txt, .pdf, or .docx)

    if ext == '.txt':
        return read_txt(blob)   # If it is a text file, read it using the text reader

    elif ext == '.pdf':
        return read_pdf(blob)   # If it is a PDF file, read it using the PDF reader

    elif ext == '.docx':
        return read_docx(blob)   # If it is a Word document, read it using the DOCX reader

    else:
        raise ValueError(f"Unsupported extension: {ext}")   # Show an error if the file type is not supported


In [ ]:
#step 9  Extract text from each uploaded file, clean it, and store it in a structured list

docs_text = []   # Create an empty list to store the cleaned text from each file

for fname,blob in docs_raw:   # Go through each uploaded file one by one
  text = load_text_by_ext(fname, blob)   # Read the file and extract its text

  text = re.sub(r'\s+\n','\n',text)   # Remove extra spaces before a new line

  text = re.sub(r'\n{3,}','\n\n',text)   # Replace multiple blank lines with just one empty line

  text = text.strip()   # Remove unwanted spaces and blank lines from the beginning and end

  docs_text.append({"name":fname,"text":text})   # Save the file name and its cleaned text into the list

for d in docs_text:   # Go through each processed document
  print(d["name"],len(d["text"]))   # Print the file name and the total number of characters in its text


In [ ]:
#step 10  Create a helper function to clean messy text

def _clean_text(s: str) -> str:   # Create a function to clean text
  s = re.sub(r'\s+', ' ', s).strip()   # Remove extra spaces
  s = re.sub(r'^\W+|\W+$', '', s).strip()   # Remove unwanted characters from the beginning and end
  return s   # Return the cleaned text

In [ ]:
#step11

import fitz   # Work with PDF files
import collections   # Store grouped data
import re   # Clean text


def _clean_text(text):
    text = re.sub(r"\s+", " ", str(text))   # Remove extra spaces
    return text.strip()   # Remove spaces from start and end


def extract_title_from_pdf_layout(pdf_bytes: bytes,
                                  consider_pages=3,
                                  top_band=0.60):   # Extract title from PDF layout

    doc = fitz.open(stream=pdf_bytes, filetype="pdf")   # Open the PDF

    if doc.page_count == 0:
        return None   # Return if PDF is empty

    spans = []   # Store text spans
    page_heights = {}   # Store page heights

    for pno in range(min(consider_pages, doc.page_count)):   # Check first few pages

        page = doc.load_page(pno)   # Load page
        page_heights[pno] = page.rect.height   # Save page height

        data = page.get_text("dict")   # Get text with layout

        for block in data.get("blocks", []):

            if "lines" not in block:
                continue   # Skip non-text blocks

            for line in block["lines"]:

                for span in line.get("spans", []):

                    text = _clean_text(span.get("text", ""))   # Clean extracted text

                    if len(text) < 3:
                        continue   # Ignore very short text

                    font_name = (span.get("font") or "").lower()   # Get font name

                    spans.append({
                        "page": pno,
                        "text": text,
                        "size": float(span.get("size", 0)),   # Save font size
                        "bold": (
                            "bold" in font_name
                            or "black" in font_name
                            or "heavy" in font_name
                        ),   # Check if text is bold
                        "y0": span.get("bbox", [0, 0, 0, 0])[1]   # Save vertical position
                    })

    if not spans:
        return None   # Return if no text found

    # Keep only upper portion of pages
    top_spans = [
        sp for sp in spans
        if (sp["y0"] / page_heights[sp["page"]]) <= top_band
    ]   # Keep text near top

    if top_spans:
        spans = top_spans

    # Remove repeated headers
    per_text_pages = collections.defaultdict(set)   # Track repeated text

    for sp in spans:
        per_text_pages[sp["text"]].add(sp["page"])

    spans = [
        sp for sp in spans
        if not (
            len(per_text_pages[sp["text"]]) >= 2
            and len(sp["text"]) <= 100
        )
    ]   # Remove repeated headers

    if not spans:
        return None

    # Group nearby lines
    groups = []   # Store grouped lines

    spans.sort(key=lambda x: (x["page"], x["y0"]))   # Sort by page and position

    for sp in spans:

        matched = False

        for g in groups:

            if (
                g["page"] == sp["page"]
                and abs(g["size"] - sp["size"]) < 1
                and g["bold"] == sp["bold"]
                and abs(sp["y0"] - g["last_y"]) < 25
            ):
                g["texts"].append(sp["text"])   # Add to existing group
                g["last_y"] = sp["y0"]
                matched = True
                break

        if not matched:
            groups.append({
                "page": sp["page"],
                "size": sp["size"],
                "bold": sp["bold"],
                "first_y": sp["y0"],
                "last_y": sp["y0"],
                "texts": [sp["text"]]
            })   # Create a new group

    candidates = []   # Store title candidates

    for g in groups:

        text = _clean_text(" ".join(g["texts"]))   # Merge grouped text

        # Ignore very short text
        if len(text) < 8:
            continue

        score = (
            g["size"] * 10
            + (20 if g["bold"] else 0)
            - (g["first_y"] * 0.02)
            + min(len(text), 120) * 0.05
        )   # Calculate title score

        candidates.append((score, text))   # Save candidate

    if candidates:
        candidates.sort(reverse=True)   # Highest score first
        return candidates[0][1]   # Return best title

    # Fallback: largest span
    best = max(
        spans,
        key=lambda sp: (
            sp["size"],
            sp["bold"],
            -sp["y0"]
        )
    )   # Choose largest text if needed

    return best["text"]   # Return final title


In [ ]:
title = extract_title_from_pdf_layout(docs_raw[0][1])   # Extract the title from the first uploaded PDF

print(title)   # Display the extracted title

In [ ]:
#step 12 title extract from docx into bytes

def extract_title_from_docx(docx_bytes: bytes):   # Extract title from a Word document
    d = DocxDocuments(io.BytesIO(docx_bytes))   # Open the DOCX file

    def paragraph_score(p):   # Score each paragraph
        text = _clean_text(p.text)   # Clean paragraph text
        if not text or len(text) < 3: return None   # Skip short text

        style_name = (p.style.name if p.style else "").lower()   # Get paragraph style

        max_size_pt, any_bold = 0.0, False   # Store font size and bold info

        for run in p.runs:   # Check each text run
            if run.font is not None:
                if run.font.size:
                    try: max_size_pt = max(max_size_pt, float(run.font.size.pt))   # Get largest font size
                    except: pass
                if run.font.bold: any_bold = True   # Check if text is bold

        style_priority = 0   # Default style priority

        if "title" in style_name: style_priority = 3   # Highest priority
        elif "heading 1" in style_name or style_name == "heading1": style_priority = 2   # Heading 1 priority
        elif "heading" in style_name: style_priority = 1   # Other headings

        return {"text": text, "style_priority": style_priority, "size": max_size_pt, "bold": any_bold}   # Return paragraph details

    candidates = []   # Store title candidates

    for p in d.paragraphs:   # Check each paragraph
        sc = paragraph_score(p)
        if sc: candidates.append(sc)   # Save valid candidate

    if not candidates: return None   # Return if no title found

    candidates.sort(key=lambda c: (c["style_priority"], c["size"], c["bold"], -len(c["text"])), reverse=True)   # Sort best candidate first

    return candidates[0]["text"]   # Return the best title

In [ ]:
print(extract_title_from_docx(docs_raw[1][1]))   # Display the title from the second uploaded Word file

In [ ]:
#step 13 Extract title from txt file

def extract_title_from_txt_firstline(txt: str):   # Extract title from a text file
    for line in txt.splitlines():   # Check each line
        if line.strip(): return _clean_text(line)   # Return the first non-empty line as the title
    return None   # Return if no title is found

In [ ]:
print(extract_title_from_txt_firstline(read_txt(docs_raw[2][1])))   # Display the title from the third uploaded text file

In [ ]:
#step 14 build the one by one title

# Build title map
name_to_title = {}   # Store file names and their titles
name_to_blob  = {fname: blob for fname, blob in docs_raw}   # Map each file to its data

for item in docs_text:   # Process each document
    fname = item["name"]; blob = name_to_blob[fname]   # Get file name and data
    ext = os.path.splitext(fname)[1].lower()   # Get file extension
    title = None   # Initialize title

    try:
        if ext == ".pdf":
            title = extract_title_from_pdf_layout(blob)   # Extract title from PDF

        elif ext == ".docx":
            title = extract_title_from_docx(blob)   # Extract title from Word file

        elif ext == ".txt":
            title = extract_title_from_txt_firstline(item["text"])   # Extract title from text file

    except Exception:
        title = None   # Ignore errors

    if not title: title = os.path.splitext(fname)[0]   # Use file name if title is not found

    name_to_title[fname] = title   # Save the title

print("Detected titles:")   # Print heading

for d in docs_text:   # Show title for each file
    print(f"• {d['name']}  →  {name_to_title[d['name']]}")   # Display file name and title

In [ ]:
#step 15 covert the text data into small chunks according to parametters

def chunk_text(text: str, target_chars: int = 1200, overlap_chars: int = 150) -> List[str]:   # Split text into small chunks
    sents = sent_tokenize(text)   # Split text into sentences
    chunks = []   # Store all chunks
    buf = ""   # Store the current chunk

    for s in sents:   # Process each sentence
        if not buf:
            buf = s   # Start a new chunk

        elif len(buf) + 1 + len(s) <= target_chars:
            buf += " " + s   # Add sentence to the current chunk

        else:
            chunks.append(buf.strip())   # Save the completed chunk

            # overlap tail
            tail = buf[-overlap_chars:]   # Keep some previous text
            buf = (tail + " " + s).strip()   # Start the next chunk with overlap

    if buf:
        chunks.append(buf.strip())   # Save the last chunk

    # filter tiny empties
    return [c for c in chunks if len(c.strip()) > 0]   # Return all valid chunks

In [ ]:
#step 16 we are creating the chunks according to passed parametters

all_docs_chunks = []   # Store chunks for all documents

for d in docs_text:   # Process each document
    ch = chunk_text(d["text"], target_chars=1400, overlap_chars=200)   # Create text chunks
    all_docs_chunks.append({"name": d["name"], "chunks": ch})   # Save document name and chunks
    print(d["name"], "→", len(ch), "chunks")   # Show number of chunks

In [ ]:
#step 17

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # Set the embedding model name
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)   # Load the embedding model

# Flatten chunks + keep metadata
corpus_texts, corpus_meta = [],[]   # Store chunk text and its information

for doc_id, d in enumerate(all_docs_chunks):   # Process each document
  for chunk_id, c in enumerate(d["chunks"]):   # Process each chunk
    corpus_texts.append(c)   # Save chunk text
    corpus_meta.append((doc_id, d["name"], chunk_id))   # Save chunk details

# Compute embedddings
BATCH = 64   # Process 64 chunks at a time

embs = []   # Store embeddings

for i in range(0, len(corpus_texts), BATCH):   # Process chunks in batches
  embs.extend(embedder.encode(corpus_texts[i:i+BATCH], normalize_embeddings=True, show_progress_bar=False))   # Convert text into embeddings

embs = np.vstack(embs).astype('float32')   # Convert embeddings into a NumPy array

# FAISS index (cosine -> use IndexFlatIP with normalized vectors)
index = faiss.IndexFlatIP(embs.shape[1])   # Create a FAISS index

index.add(embs)   # Add all embeddings to the index

print("Index build with", index.ntotal, "vectors")   # Show total indexed vectors

In [ ]:
#step 18 using pipeline function

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # Check whether GPU or CPU is available
device_id = 0 if DEVICE == "cuda" else -1   # Set the device ID

# -----------------------
# Summarizer
# -----------------------
SUMM_MODEL = "facebook/bart-large-cnn"   # Select the BART summarization model

summarizer = pipeline(
    task="text-generation",   # Create a text generation pipeline
    model=SUMM_MODEL,   # Load the summarization model
    device=device_id   # Run the model on the selected device
)

# -----------------------
# QA / Generation model
# -----------------------
GEN_MODEL = "google/flan-t5-base"   # Select the FLAN-T5 model

generator = pipeline(
    task="text-generation",   # Create a text generation pipeline
    model=GEN_MODEL,   # Load the generation model
    device=device_id   # Run the model on the selected device
)

In [ ]:
#step 19 use this function cut the sentence to the safe zone

def safe_truncate(txt, max_chars=2400):   # Limit text to a safe length
  # keep a little room
  return txt if len(txt) <= max_chars else txt[:max_chars] + "..."   # Cut long text and add "..."

In [ ]:
#step 20

def summarize_long_text(text: str, chunks_chars: int = 1200,
                        overlap: int = 150,
                        map_max_len = 256,
                        map_min_len = 60)-> str:   # Summarize a long document

#MAP: summarize each chunk with BART
  parts = chunk_text(text, target_chars=chunks_chars, overlap_chars=overlap)   # Split text into chunks

  partial_summaries = []   # Store chunk summaries

  for p in parts if parts else [text]:   # Process each chunk
      p_in = safe_truncate(p, 3500)   # Limit chunk size
      out = summarizer(p_in, max_new_tokens=map_max_len, min_length=map_min_len, do_sample=False)[0]["generated_text"]   # Summarize the chunk
      partial_summaries.append(out)   # Save the summary

# REDUCE: join partials and ask FLAN for crisp bullet points
  joined = safe_truncate(" ".join(partial_summaries), 4500)   # Combine all summaries

  reduce_prompt = f"""Summarize the document below into 7-10 concise bullet points.
each bullet must be specific and fatual, and should includ named entities (people, orgs, places), dates/years,
numbers/tables findinds if present , and concreate outcomes or clams. Avoid generic sentences.

  Document:
  \"\"\"{joined}\"\"\"

Return only bullet points, each starting with '-'."""   # Create prompt for final summary

  final = generator(reduce_prompt, max_new_tokens=512, temperature=0.2)[0]["generated_text"]   # Generate final summary

  # Light cleanup: ensure bullets
  lines = [ln.strip() for ln in final.splitlines() if ln.strip()]   # Split into lines

  bullets = []   # Store bullet points

  for ln in lines:
    if not ln.startswith("_"):
       ln = "_" + ln.lstrip(".*- ")   # Format as bullet
       bullets.append(ln)

 # Keep 7-10 bulleta if possible
  if len(bullets) < 5:
  #fallback to original joined summary split
    bullets = ['- ' + s for s in partial_summaries[:8]]   # Use backup summaries

  return "\n".join(bullets[:10]).strip()   # Return the final summary

In [ ]:
#step 21

# Re-run to rebuild summaries with the new style
doc_summaries = {}   # Store summaries for all documents

for d in docs_text:   # Process each document
    fname = d["name"]   # Get the file name
    print(f"Summarizing: {name_to_title[fname]}")   # Show which document is being summarized
    doc_summaries[fname] = summarize_long_text(d["text"])   # Generate and save the summary

print("Done.")   # Indicate that summarization is complete

In [ ]:
# Step 22: Build a search function that finds the most relevant chunks using embeddings + FAISS

def search(query: str, top_k: int = 6) -> List[Tuple[float, str, Tuple[int,str,int]]]:   # Search for the most relevant chunks

    # Convert the query into an embedding
    qe = embedder.encode([query], normalize_embeddings=True)

    # Search the FAISS index for similar chunks
    D, I = index.search(qe.astype('float32'), top_k)

    hits = []   # Store search results

    for score, idx in zip(D[0], I[0]):   # Process each result
        hits.append((float(score), corpus_texts[idx], corpus_meta[idx]))   # Save score, chunk, and metadata

    return hits   # Return the top matching chunks

In [ ]:
search("How machine learning useful to analysse data?") #searching questions

In [ ]:
# Step 23: Add safety checks: restore the answer prompt and confirm the search function exists

# === RECOVERY CELL — run this once before FAQ generation ===

# Check if the answer prompt already exists
if "ANSWER_PROMPT_TMPL" not in globals():
    ANSWER_PROMPT_TMPL = """You are a helpful assistant. Answer the question strictly using the provided context.
If the answer is not in the context, say "I don't know from the provided documents."

Context:
{context}

Question: {question}

Answer:"""   # Create the default answer prompt

# Check if the search function exists
if "search" not in globals():
    raise RuntimeError(
        "`search()` is missing. You MUST re-run the FAISS + embeddings cell that built the index and defined search()."
    )   # Stop if search() is not available

print(" ANSWER_PROMPT_TMPL restored. `search()` presence =", 'search' in globals())   # Confirm everything is ready

In [ ]:
print(globals()) #print conditions

In [ ]:
# Step 24: Create a “generic question detector” using regex (to improve FAQ quality)
# === Improved FAQ Generation v2 (varied questions, grounded answers, no placeholders) ===
# --- helpers to detect/clean generic questions ---
GENERIC_PATTERNS = [
    r"\bmain point\b",                            # ignore "What is the main point of the document?"
    r"\bwhy .* important\b",
    r"\bintended audience\b",
    r"\bscope\b",
    r"\bconclusion\b",                            # What is the conclusion?
    r"\boverview\b",
    r"\bsum(mary|marize)\b",
    r"\badditional detail\??\b",
    r"\bwhat is .*document\b",                   # Why is this document important?
    r"\bwhat does the document\b",
]
GENERIC_RE = re.compile("|".join(GENERIC_PATTERNS), re.I)  # re.IGNORECASE

In [ ]:
# Step 25: Create a “question quality checker” to keep only specific FAQ questions

def looks_specific(q: str) -> bool:   # Check if the question is specific
    q = q.strip().rstrip("?")   # Remove extra spaces and '?'

    if len(q.split()) < 6:   # Skip very short questions
        return False

    if GENERIC_RE.search(q):   # Skip generic questions
        return False

    # discourage ultra-vague starts
    if re.match(r"(?i)what is|what does|why is|who is$", q[:20]):
        return False   # Skip vague questions

    return True   # Question is specific

In [ ]:
looks_specific("what is main point of ai")   # Check if the first question is specific

looks_specific("How narrow ai help and what are some example of it?")   # Check if the second question is specific

In [ ]:
# Step 26: Remove duplicate items while keeping the original order (clean duplicates smartly)

def dedup_preserve_order(items):   # Remove duplicate items
    seen = set(); out = []   # Store seen items and unique results

    for x in items:   # Check each item
        key = re.sub(r"\s+", " ", x.strip().lower())   # Create a normalized key

        if key in seen:
            continue   # Skip duplicate items

        seen.add(key); out.append(x.strip())   # Save unique item

    return out   # Return unique items


In [ ]:
Example_list = ["apple","APPLE","banana","orange","banana","apple"]   # Sample list with duplicates

deduped_list = dedup_preserve_order(Example_list)   # Remove duplicate items

print(deduped_list)   # Display the unique items

In [ ]:
# Step 27: Generate high-quality, non-generic FAQ questions from a document summary

# --- Question generation from summary (sampling ON for variety) ---

FAQ_QS_PROMPT = """Write 12 distinct, specific FAQ questions that a reader of the following document would likely ask.
Questions must reference concrete people/organizations, locations, years/dates, metrics, methods, or outcomes from the text.
Avoid generic questions like "What is the main point?" or "Why is this important?".

Return ONLY the questions, one per line, no numbering or extra text.

Document:
\"\"\"{doc}\"\"\""""   # Prompt for generating FAQ questions


def generate_varied_questions_from_summary(summary_text: str, k: int = 12) -> list:   # Generate FAQ questions

    base = safe_truncate(summary_text, 4500)   # Limit summary length

    prompt = FAQ_QS_PROMPT.format(doc=base)   # Create the prompt

    out = generator(
        prompt,
        max_new_tokens=256,
        temperature=0.9,       # Generate varied questions
        top_p=0.9,
        do_sample=True
    )[0]["generated_text"]   # Generate questions

    qs = [q.strip().rstrip("?") + "?" for q in out.splitlines() if q.strip()]   # Format questions

    qs = dedup_preserve_order(qs)   # Remove duplicate questions

    # filter generic
    qs = [q for q in qs if looks_specific(q)]   # Keep only specific questions

    # keep top ~10 and randomize a bit to avoid same 5 every run
    random.shuffle(qs)   # Shuffle the questions

    return qs[:max(k, 5)]   # Return the required number of questions

In [ ]:
# Step 28: Answer a question using only one selected document (robust retrieval + fallback)

# --- Answering with robust context (doc retrieval + summary fallback) ---

def answer_within_doc_robust(question: str, doc_name: str, summary_text: str) -> str:   # Answer a question from a specific document

    # primary: restrict hits to this doc
    hits = search(question, top_k=12)   # Search relevant chunks

    hits = [h for h in hits if h[2][1] == doc_name]   # Keep only matching document

    ctx_blocks, seen = [], set()   # Store context and avoid duplicates

    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):   # Sort by score
        if len(" ".join(ctx_blocks)) > 4200: break   # Limit context size

        sig = hash(txt)
        if sig in seen:
            continue   # Skip duplicate chunks

        seen.add(sig)
        ctx_blocks.append(txt)   # Add chunk to context

    # fallback: if retrieval is weak, include summary
    context = "\n\n".join(ctx_blocks)   # Combine retrieved chunks

    if len(context) < 400 and summary_text:
        context = (summary_text.strip() + "\n\n" + context).strip()   # Add summary if context is small

    if not context:
        context = summary_text if summary_text else "No context."   # Final fallback

    prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)   # Create answer prompt

    ans = generator(
        prompt,
        max_new_tokens=220,
        temperature=0.2,   # Generate consistent answers
        do_sample=False
    )[0]["generated_text"].strip()   # Generate answer

    # last resort: if answer is weak, use only the summary
    if (not ans) or len(ans.split()) < 3 or "I don't know" in ans:
        if summary_text:
            prompt2 = ANSWER_PROMPT_TMPL.format(context=summary_text, question=question)   # Create fallback prompt

            ans2 = generator(prompt2, max_new_tokens=200, temperature=0.2, do_sample=False)[0]["generated_text"].strip()   # Generate fallback answer

            if ans2 and "I don't know" not in ans2:
                ans = ans2   # Use fallback answer

    return ans   # Return the final answer

In [ ]:
# Step 29: Create an FAQ generator for one document (make 5 good questions and answer them)

# --- Build FAQs using title for display, filename for retrieval mapping ---
def generate_faqs_for_document_v2(doc_name: str, title: str, text: str, summary: str) -> list:

    # If the summary is missing or too short, create a new summary from the document.
    seed = summary if (summary and len(summary) > 60) else summarize_long_text(text)

    # Generate 12 possible FAQ questions from the summary.
    candidates = generate_varied_questions_from_summary(seed, k=12)

    # If fewer than 5 questions are generated, create extra questions
    # from the summary bullet points.
    if len(candidates) < 5:
        bullets = [ln[2:].strip() for ln in seed.splitlines() if ln.strip().startswith("- ")]
        for b in bullets:
            if len(candidates) >= 5:
                break
            frag = " ".join(b.split()[:12])
            candidates.append(f"What specific findings does the document report about {frag}?")

        # Remove duplicate questions while keeping the original order.
        candidates = dedup_preserve_order(candidates)

    # Select the first 5 best questions.
    final_qs = candidates[:5]

    # Find the best answer for each selected question from the document.
    faqs = []
    for q in final_qs:
        a = answer_within_doc_robust(q, doc_name, summary)
        faqs.append((q, a))

    # Return the final list of (Question, Answer) pairs.
    return faqs

In [ ]:
# Step 30: Generate FAQs for every uploaded document and store them in a dictionary

# --- Run for all docs (uses previously computed: docs_text, name_to_title, doc_summaries) ---

# Create an empty dictionary to store FAQs for each document.
doc_faqs = {}

# Go through each uploaded document one by one.
for d in docs_text:
    # Get the document filename.
    fname = d["name"]

    # Show which document is currently being processed.
    print("Generating FAQs for", name_to_title[fname])

    # Generate FAQs for the current document and save them in the dictionary.
    doc_faqs[fname] = generate_faqs_for_document_v2(
        fname,
        name_to_title[fname],
        d["text"],
        doc_summaries.get(fname)
    )

# Display a message after FAQs have been created for all documents.
print("Done (Improved FAQ Generation v2).")

In [ ]:
# Step 31: Create the main Q&A function (retrieve relevant text, generate an answer, and return sources)

# Prompt template given to the language model.
# It tells the model to answer only from the provided context.
ANSWER_PROMPT_TMPL = """You are a helpful assistant. Answer the question strictly using the provided context.
If the answer is not in the context, say "I don't know from the provided documents."

Context:
{context}

Question: {question}

Answer:"""

# Main function to answer a user's question.
def answer_question(question: str, top_k: int = 6, max_ctx_chars: int = 4200) -> Dict:

    # Search for the most relevant chunks related to the question.
    hits = search(question, top_k=top_k)

    # Combine the retrieved chunks into one context.
    # Chunks are sorted from highest score to lowest.
    ctx_blocks = []
    seen = set()

    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):

        # Stop adding text if the context becomes too large.
        if len(" ".join(ctx_blocks)) > max_ctx_chars:
            break

        # Skip duplicate chunks.
        sig = hash(txt)
        if sig in seen:
            continue

        seen.add(sig)
        ctx_blocks.append(txt)

    # Join all selected chunks into one context string.
    context = "\n\n".join(ctx_blocks)

    # Fill the prompt with the context and user's question.
    prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)

    # Generate the final answer using the language model.
    out = generator(prompt, max_new_tokens=256, temperature=0.2)[0]["generated_text"].strip()

    # Save information about the top matching source chunks.
    sources = [{"doc_name": m[1], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:min(top_k,5)]]

    # Return the answer, source details, and a few retrieved text snippets.
    return {
        "answer": out,
        "sources": sources,
        "retrieved_snippets": [t for _,t,_ in hits[:3]]
    }

In [ ]:
# Step 32: Print a complete overview for each document (name, summary, and FAQs)

# Function to display the overview of every uploaded document.
def show_document_overview():

    # Loop through each document one by one.
    for d in docs_text:

        # Get the document filename.
        name = d["name"]

        # Print a separator line to make the output easier to read.
        print("="*100)

        # Print the document name.
        print(name)

        # Print the document summary.
        print("- Summary -")
        print(textwrap.fill(doc_summaries[name], width=100))

        # Print all FAQs related to the current document.
        print("\n- FAQs -")
        for i, (q, a) in enumerate(doc_faqs[name], 1):
            print(f"Q{i}: {q}\nA{i}: {a}\n")

# Call the function to display the overview for all documents.
show_document_overview()

In [ ]:
# Step 33: Create the Q&A controller for the Gradio UI

# ---- D) Gradio UI with Titles ----

# Create a mapping from document titles to their original filenames.
TITLE_TO_NAME = {name_to_title[d["name"]]: d["name"] for d in docs_text}

# Create a list of all document titles for the dropdown menu.
DOC_TITLES = list(TITLE_TO_NAME.keys())

# Main function that handles user questions from the Gradio interface.
def qa_interface(question, title_scope):

    # Convert the selected document title back to its original filename.
    if title_scope != "All":  # Check if the user selected a specific document instead of searching all documents.

        doc_scope = TITLE_TO_NAME[title_scope]

        # Search for relevant chunks.
        hits = search(question, top_k=12)

        # Keep only the chunks that belong to the selected document.
        hits = [h for h in hits if h[2][1] == doc_scope]

        # Combine the retrieved chunks into one context.
        ctx = []
        seen = set()

        for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):

            # Stop if the context becomes too large.
            if len(" ".join(ctx)) > 4200:
                break

            # Skip duplicate chunks.
            sig = hash(txt)
            if sig in seen:
                continue

            seen.add(sig)
            ctx.append(txt)

        # Use the collected context or show a default message if nothing is found.
        context = "\n\n".join(ctx) if ctx else "No context available."

        # Create the prompt for the language model.
        prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)

        # Generate the final answer.
        out = generator(prompt, max_new_tokens=256, temperature=0.2)[0]["generated_text"].strip()

        # Save source information for the retrieved chunks.
        sources = [{"title": name_to_title[m[1]], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:5]]

        # Return the answer and source details.
        return out, json.dumps(sources, indent=2)

    else:
        # Search across all uploaded documents.
        res = answer_question(question, top_k=8)

        # Replace document filenames with readable titles.
        for s in res["sources"]:
            s["title"] = name_to_title.get(s["doc_name"], s["doc_name"])
            del s["doc_name"]

        # Return the answer and updated source details.
        return res["answer"], json.dumps(res["sources"], indent=2)

In [ ]:
# Step 34: Show the correct summary based on the selected scope (one document or all)

# Function to return the summary based on the user's selection.
def get_summary(title_scope):

    # If "All" is selected, combine summaries from every document.
    if title_scope == "All":

        merged = []

        # Loop through each uploaded document.
        for d in docs_text:

            # Get the current document's filename.
            fname = d["name"]

            # Add the document title and its summary to the list.
            merged.append(f"### {name_to_title[fname]}\n{doc_summaries[fname]}")

        # Return all summaries as one formatted string.
        return "\n\n".join(merged)

    # Get the filename for the selected document.
    fname = TITLE_TO_NAME[title_scope]

    # Return the selected document's summary.
    # If no summary exists, return a default message.
    return doc_summaries.get(fname, "No summary available.")

In [ ]:
# Step 35: Display FAQs based on the selected scope (one document or all documents)

# Function to return FAQs based on the user's selected document.
def get_faqs(title_scope):

    # If "All" is selected, show FAQs for every document.
    if title_scope == "All":

        out = []

        # Loop through each uploaded document.
        for d in docs_text:

            # Get the current document's filename.
            fname = d["name"]

            # Get the readable title of the document.
            friendly = name_to_title[fname]

            # Add the document title and all its FAQs to the output list.
            out.append("### " + friendly + "\n" + "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(doc_faqs[fname])]))

        # Return FAQs for all documents as one formatted string.
        return "\n\n".join(out)

    # Get the filename for the selected document.
    fname = TITLE_TO_NAME[title_scope]

    # Get the FAQs for the selected document.
    faqs = doc_faqs.get(fname, [])

    # Return the formatted FAQs.
    # If there are no FAQs, return "No FAQs."
    return "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(faqs)]) or "No FAQs."

In [ ]:
# Step 36: Build the Gradio web app with 3 tabs (Q&A, Summaries, FAQs)

# Create the main Gradio application.
with gr.Blocks(title="Mini NotebookLM (Local RAG)") as demo:

    # Display the app title and a short description.
    gr.Markdown("# 📚 Mini NotebookLM (Local RAG) – Colab\nUpload ➜ Summaries & FAQs ➜ Ask Questions, grounded in your docs.")

    # Create a row for selecting the document scope.
    with gr.Row():

        # Dropdown to choose a specific document or search across all documents.
        title_scope = gr.Dropdown(DOC_TITLES, value="All", label="Document Scope (by Title)")

    # -------------------- Q&A Tab --------------------
    with gr.Tab("Ask Questions"):

        # Textbox where the user enters a question.
        question = gr.Textbox(label="Your question", placeholder="Ask something grounded in your uploaded docs…")

        # Button to generate an answer.
        btn = gr.Button("Answer")

        # Textbox to display the generated answer.
        answer = gr.Textbox(label="Answer", lines=8)

        # Textbox to display the source chunks used for the answer.
        sources = gr.Textbox(label="Top Source Chunks (title, chunk_id, score)", lines=8)

        # Connect the button with the Q&A function.
        btn.click(qa_interface, inputs=[question, title_scope], outputs=[answer, sources])

    # -------------------- Summary Tab --------------------
    with gr.Tab("Summaries"):

        # Button to show the summary.
        btn_sum = gr.Button("Show Summary")

        # Textbox to display the summary.
        sum_box = gr.Textbox(label="Summary", lines=20)

        # Connect the button with the summary function.
        btn_sum.click(get_summary, inputs=[title_scope], outputs=[sum_box])

    # -------------------- FAQ Tab --------------------
    with gr.Tab("FAQs"):

        # Button to show the FAQs.
        btn_faq = gr.Button("Show FAQs (5)")

        # Textbox to display the FAQs.
        faq_box = gr.Textbox(label="FAQs", lines=20)

        # Connect the button with the FAQ function.
        btn_faq.click(get_faqs, inputs=[title_scope], outputs=[faq_box])

# Launch the Gradio web application and create a public shareable link.
demo.launch(share=True)